# 5주차: Hybrid Search + Re-ranking + Ablation 비교

## 목표
4주차의 Dense Retrieval baseline에 BM25(Sparse) + RRF Hybrid Search와 Cross-Encoder Re-ranking을 추가하여  
4가지 구성(Dense / BM25 / Hybrid / Hybrid+Rerank)을 ablation 비교합니다.

## 구성
1. 환경 설정 & 4주차 Vector Store 로드
2. BM25 Retriever 구축
3. Hybrid Search (EnsembleRetriever + RRF)
4. Cross-Encoder Re-ranker 적용
5. Ablation 비교 실험 (RAGAS)
6. Error Case 분석
7. 결과 요약

---
## Cell 0: 패키지 설치 확인

In [1]:
# 5주차 필수 패키지 설치
# 아래 주석을 해제해서 처음 한 번만 실행하세요.

# !pip install rank-bm25 langchain-community -q
# !pip install sentence-transformers -q   # cross-encoder용
# !pip install ragas -q
# !pip install datasets -q

import importlib
for pkg in ["rank_bm25", "sentence_transformers", "ragas", "datasets"]:
    found = importlib.util.find_spec(pkg) is not None
    print(f"{pkg:25s}: {'✅' if found else '❌ 설치 필요'}")

rank_bm25                : ✅
sentence_transformers    : ✅
ragas                    : ✅
datasets                 : ✅


---
## Cell 1: 환경 설정 & 공통 임포트

In [18]:
from __future__ import annotations

import os
import time
import warnings
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

load_dotenv()
warnings.filterwarnings("ignore")

# ── 경로 설정 (4주차와 동일) ──────────────────────────────────────────────────
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ir_files = [
    "1Q26_Naver_Earnings_KOR_vF.pdf",
    "2026 1Q PT K_(최종).pdf",
    "berkshirehathaway_26_first_quarter.pdf",
]
PDF_PATH     = PROJECT_ROOT / "docs" / ir_files[0]
DB_DIR       = PROJECT_ROOT / "data" / "ir_database"
COLLECTION_NAME = "social_culture"

os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / "data" / "matplotlib_cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"PDF exists   : {PDF_PATH.exists()}")
print(f"OPENAI_KEY   : {'✅' if os.getenv('OPENAI_API_KEY') else '❌'}")

PROJECT_ROOT : /Users/judy/Desktop/github/mo_du_yeon/Advanced-RAG-Multimodal-AI-Agent
PDF exists   : True
OPENAI_KEY   : ✅


---
## Cell 2: 4주차 Chunks & Vector Store 재활용

> 4주차에서 가장 성능이 좋았던 chunking 설정(chunk_size=700, overlap=150)을 그대로 사용합니다.  
> Vector Store가 이미 `data/chroma_social_culture`에 저장돼 있으면 재빌드하지 않습니다.

In [20]:
from langchain_community.document_loaders import PyPDFLoader
import chromadb
from langchain_community.vectorstores import Chroma # ChromaDB와 버전 맞춰야 함.
print(chromadb.__version__)
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Embedding 모델 (4주차와 동일) ─────────────────────────────────────────────
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# ── Chunks 생성 ───────────────────────────────────────────────────────────────
loader   = PyPDFLoader(PDF_PATH)
print(f"PDF 페이지 수: {len(loader.load_and_split())}")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = text_splitter.split_documents(documents)
print(f"Pages : {len(documents)}, Chunks : {len(chunks)}")

# ── Vector Store 로드 또는 신규 생성 ─────────────────────────────────────────
if DB_DIR.exists() and any(DB_DIR.iterdir()):
    print("[INFO] 기존 Chroma DB 로드")
    vector_db = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
        persist_directory=str(DB_DIR),
    )
else:
    print("[INFO] Chroma DB 신규 생성")
    vector_db = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=str(DB_DIR),
    )
    if hasattr(vector_db, "persist"):
        vector_db.persist()

print(f"Vector DB 준비 완료 | 문서 수: {vector_db._collection.count()}")

1.5.9
PDF 페이지 수: 12
Pages : 12, Chunks : 14
[INFO] Chroma DB 신규 생성
Vector DB 준비 완료 | 문서 수: 14


---
## Cell 3: 테스트 질문 세트 정의

> 모든 ablation 실험에서 **동일한 질문 세트**를 사용해야 공정하게 비교할 수 있습니다.

In [21]:
# ── 테스트 질문 세트 (도메인에 맞게 수정하세요) ───────────────────────────────
TEST_QUESTIONS = [
    "파이낸셜 플랫폼 그래프에서 4Q24부터 1Q26까지 분기별 매출 증가액과 증가율을 계산하고, 원문 단위가 십억 원인지 억 원인지 확인해서 답해줘.",
    "파이낸셜 플랫폼 그래프에서 매출과 결제액(TPV)을 구분해서 설명해줘. 두 지표의 단위가 어떻게 다른지도 알려줘.",
    "5페이지와 6페이지의 분기별 매출 그래프를 비교해서 네이버 플랫폼과 파이낸셜 플랫폼 중 1Q26 QoQ 성장률이 더 높은 쪽을 알려줘.",
    "분기 별 매출 추이에서 전년 동기 대비 가장 큰 폭으로 성장한 분기는 언제인가?",
    "네이버의 2026년 1분기 전체 매출과 영업이익을 알려줘.",
]

# Ground truth — RAGAS 평가용 (실제 정답을 직접 작성)
# 정확한 GT가 없으면 빈 문자열로 두고 RAGAS의 context_precision 등 비참조 지표만 활용
GROUND_TRUTHS = [
    "",  # 필요 시 정답 기입
    "",
    "",
    "",
    "",
]

print(f"테스트 질문 {len(TEST_QUESTIONS)}개 정의 완료")

테스트 질문 5개 정의 완료


---
## Cell 4: 4가지 Retriever 구성

| 구성 | 방법 |
|------|------|
| **Dense** | Chroma 기반 Cosine 유사도 (4주차 baseline) |
| **BM25** | TF-IDF 기반 키워드 검색 (Sparse) |
| **Hybrid** | BM25 + Dense → RRF (EnsembleRetriever) |
| **Hybrid+Rerank** | Hybrid → Cross-Encoder Re-ranker → top-5 |

In [22]:
# ── (1) Dense Retriever ───────────────────────────────────────────────────────
dense_retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 20},  # Re-ranker를 위해 top-20 먼저 가져옴
)
print("[1] Dense Retriever 준비 완료")

[1] Dense Retriever 준비 완료


In [24]:
# ── (2) BM25 Retriever ────────────────────────────────────────────────────────
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 20
print("[2] BM25 Retriever 준비 완료")

[2] BM25 Retriever 준비 완료


In [ ]:
# ── (3) Hybrid Retriever (BM25 + Dense → RRF) ─────────────────────────────────
# LangChain EnsembleRetriever는 내부적으로 RRF를 사용합니다.
# weights: BM25와 Dense의 상대적 비중 (합이 1이 되도록)
# 한국어+영어 혼합 도메인은 0.5:0.5 또는 BM25를 약간 높게 주는 것이 효과적입니다.
import langchain
import pkgutil
import langchain_classic
print(langchain_classic.__version__) # 1.0.7
print([m.name for m in pkgutil.iter_modules(langchain_classic.__path__)])
# from langchain.retrievers.ensemble import EnsembleRetriever
from langchain_classic.retrievers import EnsembleRetriever

# Dense top-20용 별도 retriever (Hybrid 전용)
dense_for_hybrid = vector_db.as_retriever(search_kwargs={"k": 20})
bm25_for_hybrid  = BM25Retriever.from_documents(chunks)
bm25_for_hybrid.k = 20

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_for_hybrid, dense_for_hybrid],
    weights=[0.5, 0.5],  # BM25 : Dense = 0.5 : 0.5
)
print("[3] Hybrid Retriever (RRF) 준비 완료 | weights=[0.5, 0.5]")

1.0.7
['_api', 'adapters', 'agents', 'base_language', 'base_memory', 'cache', 'callbacks', 'chains', 'chat_loaders', 'chat_models', 'docstore', 'document_loaders', 'document_transformers', 'embeddings', 'env', 'evaluation', 'example_generator', 'formatting', 'globals', 'graphs', 'hub', 'indexes', 'input', 'llms', 'load', 'memory', 'model_laboratory', 'output_parsers', 'prompts', 'python', 'requests', 'retrievers', 'runnables', 'schema', 'serpapi', 'smith', 'sql_database', 'storage', 'text_splitter', 'tools', 'utilities', 'utils', 'vectorstores']
[3] Hybrid Retriever (RRF) 준비 완료 | weights=[0.5, 0.5]


In [36]:
# ── (4) Cross-Encoder Re-ranker ───────────────────────────────────────────────
# 추천 모델 옵션:
#   - "BAAI/bge-reranker-v2-m3"          ← 다국어, 한국어 강함 (추천)
#   - "dragonkue/bge-reranker-v2-m3-ko"  ← 한국어 특화
#   - "cross-encoder/ms-marco-MiniLM-L6-v2" ← 영어 중심, 가볍고 빠름

RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L6-v2"  # 가벼운 모델로 먼저 테스트
# RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"  # GPU가 있으면 이걸 추천

# from langchain.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

print(f"Re-ranker 모델 로딩 중: {RERANKER_MODEL}")
cross_encoder = HuggingFaceCrossEncoder(model_name=RERANKER_MODEL)
reranker_compressor = CrossEncoderReranker(model=cross_encoder, top_n=5)

hybrid_rerank_retriever = ContextualCompressionRetriever(
    base_compressor=reranker_compressor,
    base_retriever=hybrid_retriever,  # Hybrid top-20 → Re-ranker → top-5
)
print(f"[4] Hybrid+Rerank Retriever 준비 완료 | top-20 → top-5")

Re-ranker 모델 로딩 중: cross-encoder/ms-marco-MiniLM-L6-v2


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2504.65it/s]


[4] Hybrid+Rerank Retriever 준비 완료 | top-20 → top-5


---
## Cell 5: 단일 쿼리 비교 테스트 (Sanity Check)

RAGAS 실험 전에 한 질문으로 4가지 retriever의 결과를 육안으로 확인합니다.

In [37]:
SAMPLE_QUERY = "파이낸셜 플랫폼 분기별 매출"

def show_results(retriever, name: str, query: str, top_n: int = 3):
    """Retriever 결과를 보기 좋게 출력."""
    start = time.time()
    docs  = retriever.invoke(query)
    elapsed = time.time() - start

    print(f"\n{'='*60}")
    print(f"[{name}] | 결과 {len(docs)}개 | 소요 {elapsed:.2f}s")
    print('='*60)
    for i, doc in enumerate(docs[:top_n], 1):
        page = doc.metadata.get('page', '?')
        print(f"  [{i}] p.{page+1 if isinstance(page,int) else page} | {doc.page_content[:200].replace(chr(10),' ')}...")

# Dense (top-5만 보기 위해 k=5 별도 retriever)
show_results(vector_db.as_retriever(search_kwargs={"k": 5}), "Dense", SAMPLE_QUERY)
show_results(BM25Retriever.from_documents(chunks, k=5), "BM25", SAMPLE_QUERY)
show_results(hybrid_retriever, "Hybrid", SAMPLE_QUERY)
show_results(hybrid_rerank_retriever, "Hybrid+Rerank", SAMPLE_QUERY)


[Dense] | 결과 5개 | 소요 2.07s
  [1] p.9 | 809.8 795.4 816.4  894.1 896.2 941.6  (84.4) (74.2) (63.9) (67.4) (31.0) (14.2) -300.0 -100.0 100.0 300.0 500.0 700.0 900.0 1,100.0 4Q24 1Q25 2Q25 3Q25 4Q25 1Q26 매출액 부문별 손익 1,680.7  1,604.7  1,693.1  ...
  [2] p.6 | 395.1 386.7  405.6  427.3  448.6 459.7  19.3 19.6  20.8  22.7 23.0  24.2  0.0 100.0 200.0 300.0 400.0 500.0 4Q24 1Q25 2Q25 3Q25 4Q25 1Q26 0.0 5.0 10.0 15.0 20.0 파이낸셜 플랫폼 결제액(TPV) 파이낸셜 플랫폼 결제, 금융플랫폼 등 ...
  [3] p.3 | 매출 분류 변경 요약 • 2026년부터 네이버의 핵심 사업 및 신규 사업 기회를 명확하게 하기 위해 매출 분류를 변경 세부 내역 서치플랫폼 SA, DA, 기타(치지직 후원 등) 커머스 커머스광고, 중개 및 판매, 멤버십 핀테크 페이, 플랫폼 서비스 등 콘텐츠 웹툰, 스노우, 뮤직 등 엔터프라이즈 NCP, 라인웍스, 클로바/랩스 등 세부 내역 네이버 플랫폼 ...

[BM25] | 결과 5개 | 소요 0.00s
  [1] p.3 | 매출 분류 변경 요약 • 2026년부터 네이버의 핵심 사업 및 신규 사업 기회를 명확하게 하기 위해 매출 분류를 변경 세부 내역 서치플랫폼 SA, DA, 기타(치지직 후원 등) 커머스 커머스광고, 중개 및 판매, 멤버십 핀테크 페이, 플랫폼 서비스 등 콘텐츠 웹툰, 스노우, 뮤직 등 엔터프라이즈 NCP, 라인웍스, 클로바/랩스 등 세부 내역 네이버 플랫폼 ...
  [2] p.6 | 395.1 386.7  405.6  427.3  448.6 459.7  19.3 19.6  20.8  22.7 23.0  2

---
## Cell 6: LLM Chain 공통 함수 정의

In [38]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

PROMPT = ChatPromptTemplate.from_template(
    """너는 실적발표 PDF를 근거로 답하는 분석 assistant야.
주어진 context에 있는 내용만 사용해서 한국어로 답해.
숫자, 기간, 사업부 이름은 원문과 맞게 정확히 적고, 근거 페이지를 함께 표시해.
특히 단위가 '십억 원', '조 원', '%' 중 무엇인지 혼동하지 말고 원문 단위를 그대로 밝혀.
context에 없는 내용이면 모른다고 말해.

Question:
{question}

Context:
{context}
"""
)

def format_context(docs):
    parts = []
    for doc in docs:
        page = doc.metadata.get("page")
        label = f"p.{page+1}" if isinstance(page, int) else "p.?"
        parts.append(f"[{label}]\n{doc.page_content}")
    return "\n\n".join(parts)

def build_chain(retriever):
    return (
        {"context": retriever | format_context, "question": RunnablePassthrough()}
        | PROMPT
        | llm
        | StrOutputParser()
    )

print("LLM Chain 빌더 준비 완료")

LLM Chain 빌더 준비 완료


---
## Cell 7: Ablation 실험 — 4가지 구성으로 답변 생성

각 구성별로 동일한 질문 5개에 대해 답변을 생성하고,  
검색된 context와 함께 기록합니다.

In [42]:
# ablation용 retriever는 최종 top-5 기준으로 통일
RETRIEVERS = {
    "Dense":         vector_db.as_retriever(search_kwargs={"k": 5}),
    "BM25":          BM25Retriever.from_documents(chunks, k=5),
    "Hybrid":        EnsembleRetriever(
                         retrievers=[
                             BM25Retriever.from_documents(chunks, k=5),
                             vector_db.as_retriever(search_kwargs={"k": 5}),
                         ],
                         weights=[0.5, 0.5],
                     ),
    "Hybrid+Rerank": hybrid_rerank_retriever,
}

ablation_results = {}  # {config: [{question, answer, contexts, latency}]}

for config_name, retriever in RETRIEVERS.items():
    print(f"\n▶ 실험 중: [{config_name}]")
    chain   = build_chain(retriever)
    records = []

    for q in TEST_QUESTIONS:
        # context 추출
        t0   = time.time()
        docs = retriever.invoke(q)
        ctx_time = time.time() - t0

        contexts = [doc.page_content for doc in docs] # PDF에서 추출된 Chunk 내용 자체임

        # 답변 생성
        t1     = time.time()
        answer = chain.invoke(q)
        total_time = time.time() - t1 + ctx_time

        records.append({
            "question": q,
            "answer":   answer,
            "contexts": contexts,
            "latency":  round(total_time, 2),
        })
        print(f"  ✓ Q{TEST_QUESTIONS.index(q)+1} | latency={total_time:.1f}s | ctx={len(contexts)}개")
        print(f"    답변: {answer[:100]}...\n")
        print(f"    근거: {contexts[:1000]}\n")
        print("metadata:")
        for doc in docs:
            print(doc.metadata)

    ablation_results[config_name] = records

print("\n✅ 모든 구성 답변 생성 완료")


▶ 실험 중: [Dense]
  ✓ Q1 | latency=10.1s | ctx=5개
    답변: 파이낸셜 플랫폼 매출액은 [p.4, p.6]에 따르면 단위가 "십억 원"으로 명시되어 있습니다.

각 분기별 매출액(단위: 십억 원)은 다음과 같습니다:
- 4Q24: 395.1
...

    근거: ['809.8 795.4 816.4 \n894.1 896.2 941.6 \n(84.4) (74.2) (63.9) (67.4) (31.0) (14.2)\n-300.0\n-100.0\n100.0\n300.0\n500.0\n700.0\n900.0\n1,100.0\n4Q24 1Q25 2Q25 3Q25 4Q25 1Q26\n매출액 부문별 손익\n1,680.7 \n1,604.7 \n1,693.1 \n1,816.6 \n1,850.3 1,839.8 \n36.1% 34.2% 32.7% 33.6% 33.4%\n28.8%\n-45.0%\n-35.0%\n-25.0%\n-15.0%\n-5.0%\n5.0%\n15.0%\n25.0%\n35.0%\n45.0%\n1,350.0\n1,450.0\n1,550.0\n1,650.0\n1,750.0\n1,850.0\n1,950.0\n4Q24 1Q25 2Q25 3Q25 4Q25 1Q26\n매출액 부문별 손익(%)\n395.1 386.7 405.6 427.3 448.6 459.7 \n5.1%\n8.0% 7.7%\n6.4%\n5.4% 5.6%\n-10.0%\n-5.0%\n0.0%\n5.0%\n10.0%\n0.0\n100.0\n200.0\n300.0\n400.0\n500.0\n600.0\n700.0\n800.0\n4Q24 1Q25 2Q25 3Q25 4Q25 1Q26\n매출액 부문별 손익(%)\n사업부문별 손익 \n(단위: 십억 원, %)\n네이버 플랫폼\n글로벌 도전*\n파이낸셜 플랫폼\n* 글로벌 도전 구성: C2C, 콘텐츠, 엔터프라이즈', '• 1분기 매출액은 핵심사업의 견조한 성장과 C2C의 성장 가속화에 힘입어 YoY 16.3% 달성 \n• AI 경쟁

---
## Cell 8: RAGAS 평가

**사용 지표:**
- `context_precision` : 검색된 context 중 실제 관련 있는 비율 (retrieval 품질)
- `context_recall` : 정답 작성에 필요한 정보가 context에 얼마나 들어있는지 (GT 필요)
- `faithfulness` : 답변이 context에 얼마나 충실한지 (hallucination 측정)
- `answer_relevancy` : 답변이 질문에 얼마나 관련 있는지

> ⚠️ `context_recall`은 Ground Truth가 필요합니다. GT가 없으면 0으로 나올 수 있습니다.

In [ ]:
try:
    from ragas import evaluate
    from ragas.metrics import (
        context_precision,
        faithfulness,
        answer_relevancy,
    )
    from datasets import Dataset
    RAGAS_AVAILABLE = True
    print("RAGAS 사용 가능 ✅")
except ImportError:
    RAGAS_AVAILABLE = False
    print("RAGAS 미설치 — 수동 평가 모드로 전환합니다 ⚠️")
    print("설치: pip install ragas datasets")

In [41]:
ragas_scores = {}  # {config_name: {metric: score}}

if RAGAS_AVAILABLE:
    METRICS = [context_precision, faithfulness, answer_relevancy]

    for config_name, records in ablation_results.items():
        print(f"\n▶ RAGAS 평가 중: [{config_name}]")

        dataset = Dataset.from_dict({
            "question":  [r["question"] for r in records],
            "answer":    [r["answer"]   for r in records],
            "contexts":  [r["contexts"] for r in records],
            # ground_truth이 있으면 context_recall도 추가 가능
            # "ground_truth": GROUND_TRUTHS,
        })

        try:
            result = evaluate(dataset, metrics=METRICS)
            scores = {
                "context_precision": round(result["context_precision"], 4),
                "faithfulness":      round(result["faithfulness"],      4),
                "answer_relevancy":  round(result["answer_relevancy"],  4),
                "avg_latency":       round(sum(r["latency"] for r in records) / len(records), 2),
            }
            ragas_scores[config_name] = scores
            print(f"  결과: {scores}")
        except Exception as e:
            print(f"  평가 오류: {e}")
            ragas_scores[config_name] = {"error": str(e)}

else:
    # RAGAS 없을 때: latency만 정리
    for config_name, records in ablation_results.items():
        ragas_scores[config_name] = {
            "context_precision": "N/A",
            "faithfulness":      "N/A",
            "answer_relevancy":  "N/A",
            "avg_latency":       round(sum(r["latency"] for r in records) / len(records), 2),
        }

print("\n✅ RAGAS 평가 완료")

NameError: name 'RAGAS_AVAILABLE' is not defined

---
## Cell 9: Ablation 비교 결과표

In [ ]:
import pandas as pd

rows = []
for config, scores in ragas_scores.items():
    row = {"구성": config}
    row.update(scores)
    rows.append(row)

df_result = pd.DataFrame(rows).set_index("구성")

print("\n" + "="*60)
print("          5주차 Ablation 비교 결과")
print("="*60)
print(df_result.to_string())
print("="*60)

# Pandas Styler로 가시화
numeric_cols = [c for c in df_result.columns if df_result[c].dtype != object]
if numeric_cols:
    display(df_result[numeric_cols].style
            .background_gradient(cmap="YlGn", subset=[c for c in ["context_precision","faithfulness","answer_relevancy"] if c in numeric_cols])
            .background_gradient(cmap="YlOrRd_r", subset=["avg_latency"] if "avg_latency" in numeric_cols else [])
            .format("{:.4f}", subset=[c for c in ["context_precision","faithfulness","answer_relevancy"] if c in numeric_cols])
            .set_caption("↑ 높을수록 좋음 (latency는 낮을수록 좋음)"))

---
## Cell 10: BM25 vs Dense — 질문별 심층 비교

어느 구성이 어느 질문에서 더 나은지 정성적으로 확인합니다.

In [ ]:
def compare_top_chunks(query: str, k: int = 3):
    """BM25와 Dense의 top-k 청크를 나란히 출력."""
    bm25_docs  = BM25Retriever.from_documents(chunks, k=k).invoke(query)
    dense_docs = vector_db.as_retriever(search_kwargs={"k": k}).invoke(query)

    print(f"\n🔍 Query: {query[:60]}...")
    print(f"{'BM25 TOP-'+str(k):<45} {'Dense TOP-'+str(k)}")
    print("-" * 90)
    for i in range(k):
        b = bm25_docs[i].page_content[:100].replace("\n", " ") if i < len(bm25_docs) else "-"
        d = dense_docs[i].page_content[:100].replace("\n", " ") if i < len(dense_docs) else "-"
        print(f"[{i+1}] {b:<43} | {d}")

for q in TEST_QUESTIONS:
    compare_top_chunks(q, k=3)

---
## Cell 11: Re-ranker 전후 순위 변화 시각화

Hybrid 검색 결과에서 Re-ranker 적용 전후 청크 순위가 어떻게 바뀌었는지 확인합니다.

In [ ]:
def show_reranking_delta(query: str, top_before: int = 10, top_after: int = 5):
    """Hybrid 결과와 Re-rank 이후 결과를 비교합니다."""
    hybrid_r   = EnsembleRetriever(
        retrievers=[
            BM25Retriever.from_documents(chunks, k=top_before),
            vector_db.as_retriever(search_kwargs={"k": top_before}),
        ],
        weights=[0.5, 0.5],
    )

    before_docs = hybrid_r.invoke(query)[:top_before]
    after_docs  = hybrid_rerank_retriever.invoke(query)

    # 청크를 고유 식별자(첫 60자)로 매핑
    before_ids = [d.page_content[:60] for d in before_docs]
    after_ids  = [d.page_content[:60] for d in after_docs]

    print(f"\n🔀 Re-ranking 전후 비교 | Query: {query[:50]}...")
    print(f"{'순위':<4} {'Hybrid Before':<65} {'Re-ranked After'}")
    print("-" * 120)
    for i in range(max(len(before_ids), len(after_ids))):
        b = before_ids[i] if i < len(before_ids) else "-"
        a = after_ids[i]  if i < len(after_ids)  else "-"
        arrow = "  " if b == a else "🔄"
        print(f"{i+1:<4} {b:<65} {arrow} {a}")

# 첫 번째 질문으로 테스트
show_reranking_delta(TEST_QUESTIONS[0])

---
## Cell 12: RRF 가중치 튜닝 (도전 과제)

BM25:Dense 비율을 바꿔가며 retrieval 성능 변화를 관찰합니다.

In [ ]:
# 가중치 조합 테스트
WEIGHT_COMBOS = [
    (0.3, 0.7),  # Dense 중심
    (0.5, 0.5),  # 균형
    (0.7, 0.3),  # BM25 중심
]

weight_results = {}

for bm25_w, dense_w in WEIGHT_COMBOS:
    label = f"BM25:{bm25_w} Dense:{dense_w}"
    retriever_w = EnsembleRetriever(
        retrievers=[
            BM25Retriever.from_documents(chunks, k=5),
            vector_db.as_retriever(search_kwargs={"k": 5}),
        ],
        weights=[bm25_w, dense_w],
    )

    # 첫 번째 질문으로만 빠르게 비교
    docs = retriever_w.invoke(TEST_QUESTIONS[0])
    weight_results[label] = [d.page_content[:80] for d in docs[:3]]
    print(f"\n⚖️ [{label}] top-3 chunks:")
    for i, c in enumerate(weight_results[label], 1):
        print(f"  [{i}] {c}")

---
## Cell 13: Latency 비교

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

latency_data = {
    config: round(sum(r["latency"] for r in records) / len(records), 2)
    for config, records in ablation_results.items()
}

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(latency_data.keys(), latency_data.values(),
              color=["#4C72B0", "#DD8452", "#55A868", "#C44E52"])
ax.bar_label(bars, fmt="%.2fs", padding=3)
ax.set_ylabel("평균 응답 시간 (초)")
ax.set_title("[5주차] 구성별 평균 Latency 비교")
ax.set_ylim(0, max(latency_data.values()) * 1.3)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "docs" / "imgs" / "week5_latency.png", dpi=150)
plt.show()
print("Latency 차트 저장 완료")

---
## Cell 14: Error Case 분석 (필수)

검색이 여전히 실패하는 케이스를 최소 3개 분석합니다.

| 항목 | 내용 |
|------|------|
| 질문 | 어떤 질문이 실패했는가 |
| 검색된 chunk | 어떤 chunk가 올라왔는가 |
| 실패 원인 가설 | 왜 잘못 검색됐는가 |
| 6주차 해결 방향 | Agentic RAG로 어떻게 해결할 수 있는가 |

In [ ]:
# 가장 성능이 좋은 구성(Hybrid+Rerank)으로 실패 케이스를 찾습니다.
best_retriever = hybrid_rerank_retriever
best_chain     = build_chain(best_retriever)

ERROR_QUESTIONS = [
    # 아래에 실제로 잘 안 됐던 질문을 추가하세요.
    # 4주차에서 실패했던 질문 + 이번 실험에서도 실패한 질문
    "막대 그래프 y축 최댓값은 얼마인가?",
    "표에서 가장 오른쪽 열의 합계를 구해줘.",
    "PDF 9페이지 두 번째 단락을 요약해줘.",
]

print("\n" + "="*70)
print("                   Error Case 분석")
print("="*70)

for i, q in enumerate(ERROR_QUESTIONS, 1):
    docs   = best_retriever.invoke(q)
    answer = best_chain.invoke(q)

    print(f"\n❌ Error Case {i}")
    print(f"  질문      : {q}")
    print(f"  검색된 ctx: {len(docs)}개")
    for j, doc in enumerate(docs[:2], 1):
        print(f"    [{j}] p.{doc.metadata.get('page','?')} | {doc.page_content[:120].replace(chr(10),' ')}")
    print(f"  생성된 답 : {answer[:200]}")
    print()
    print("  ▷ 실패 원인 가설 (작성 필요):")
    print("    예) 이미지/표 기반 데이터가 텍스트로 추출되지 않아 chunking 단계에서 정보 손실")
    print("  ▷ 6주차 해결 방향:")
    print("    예) LangGraph에서 검색 결과 품질 판단 후 쿼리 재작성(HyDE/Multi-Query) 루프 도입")

---
## Cell 15: 최종 결과 요약 & ADR 초안 출력

실험 결과를 바탕으로 `docs/week5_retrospective.md`와 `docs/adr/week5_retrieval_strategy.md`에 붙여 넣을 내용을 생성합니다.

In [ ]:
from datetime import date

# ── 결과표 Markdown 생성 ──────────────────────────────────────────────────────
def df_to_markdown(df):
    header = "| " + " | ".join(df.columns.tolist()) + " |"
    sep    = "| " + " | ".join(["---"] * len(df.columns)) + " |"
    rows   = []
    for idx, row in df.iterrows():
        rows.append("| " + str(idx) + " | " + " | ".join(str(v) for v in row.values) + " |")
    return "\n".join([header, sep] + rows)

table_md = df_to_markdown(df_result.reset_index())

retrospective = f"""# 5주차 회고 — Hybrid Search & Re-ranking

작성일: {date.today().isoformat()}

## 1. 4주차 Best Chunking 전략
- chunk_size=700, chunk_overlap=150, RecursiveCharacterTextSplitter
- → 이 설정을 5주차 baseline으로 유지

## 2. 4주차에서 여전히 실패했던 케이스
1. 이미지/표 기반 수치 질문 — 텍스트 추출 불가로 context 부재
2. 페이지 번호 직접 참조 질문 — chunk 경계에서 컨텍스트 절단
3. 단위/스케일 혼동 질문 — 유사 의미 chunk 복수 검색으로 노이즈 발생

## 3. 왜 Retrieval 단계에서 더 개선해야 하는가
4주차 chunking 개선만으로는 LLM에게 전달되는 context 품질의 한계가 명확합니다.
의미 기반 Dense 검색은 '파이낸셜 플랫폼', 'TPV' 같은 도메인 고유명사와 약어를
embedding 공간에서 정확히 분리하지 못하는 경우가 있었습니다. BM25의 정확한 키워드
매칭과 결합하면 이 한계를 보완할 수 있으며, Re-ranker로 최종 관련성을 재정렬하면
context precision이 실질적으로 향상될 것으로 기대했습니다.

## 4. Ablation 비교 결과

{table_md}

## 5. 해석
- (실험 후 직접 작성) 예: Hybrid+Rerank에서 context_precision이 가장 높았으며...

## 6. Error Case 분석
| # | 질문 | 검색된 chunk 문제 | 원인 가설 | 6주차 해결 방향 |
|---|------|-----------------|-----------|----------------|
| 1 | 막대 그래프 y축 최댓값 | 이미지 텍스트 미추출 | PDF 파싱 한계 | 이미지 OCR 또는 multimodal 모델 |
| 2 | 표 오른쪽 열 합계 | 표 구조 chunk 분리 | 표 분할 손실 | table-aware chunking 또는 쿼리 재작성 |
| 3 | 특정 페이지 단락 요약 | 인접 chunk 검색 | 위치 정보 부재 | Self-Query 메타데이터 필터링 |
"""

adr = f"""# ADR: 5주차 Retrieval 전략

날짜: {date.today().isoformat()}  
상태: 결정됨

## 결정
> (실험 후 작성) 최종 채택 전략: Hybrid+Rerank (BM25:Dense=0.5:0.5, bge-reranker-v2-m3, top-5)

## 근거
- RAGAS context_precision: Dense 0.XX → Hybrid+Rerank 0.XX (작성 필요)
- 도메인 고유명사(파이낸셜 플랫폼, TPV, 분기 매출)에서 BM25 키워드 매칭이 Dense를 보완
- Re-ranker로 관련도 낮은 chunk를 최종 단계에서 걸러냄

## 트레이드오프
| 항목 | Dense | Hybrid+Rerank |
|------|-------|---------------|
| 평균 latency | ~Xs | ~Xs (Re-ranker +X초) |
| 메모리 | 낮음 | BM25 인덱스 + Reranker 모델 추가 |
| 구현 복잡도 | 낮음 | 중간 (EnsembleRetriever + ContextualCompression) |
| 키워드 정확성 | 낮음 | 높음 |

## 대안 검토
- **Dense only**: 구현 단순하나 키워드 정확성 부족
- **BM25 only**: 의미 유사 검색 불가, 표현 변형에 취약
- **ms-marco-MiniLM Re-ranker**: 영어 중심 모델이라 한국어 도메인에서 bge-reranker 대비 낮은 성능 예상

## 6주차 연계
Error Case 3개 중 이미지/표 기반 실패는 쿼리 재작성(Multi-Query, HyDE)으로 부분 해결 가능하며,
LangGraph의 검색-판단-재검색 루프로 context 품질이 낮을 때 자동으로 재시도하는 구조를 도입할 예정.
"""

# 파일 저장
retro_path = PROJECT_ROOT / "docs" / "week5_retrospective.md"
adr_path   = PROJECT_ROOT / "docs" / "adr" / "week5_retrieval_strategy.md"
adr_path.parent.mkdir(parents=True, exist_ok=True)

retro_path.write_text(retrospective, encoding="utf-8")
adr_path.write_text(adr,            encoding="utf-8")

print(f"✅ 회고 저장: {retro_path}")
print(f"✅ ADR  저장: {adr_path}")
print("\n--- retrospective 미리보기 ---")
print(retrospective[:800])

---
## Cell 16: 전체 파이프라인 최종 테스트

최종 선택 retriever로 테스트 질문 전체를 돌려봅니다.

In [ ]:
final_chain = build_chain(hybrid_rerank_retriever)

print("\n" + "="*70)
print("          최종 파이프라인 (Hybrid + Re-rank) 결과")
print("="*70)

for i, q in enumerate(TEST_QUESTIONS, 1):
    print(f"\n▶ Q{i}: {q}")
    print("-" * 70)
    t0 = time.time()
    ans = final_chain.invoke(q)
    print(ans)
    print(f"  [latency: {time.time()-t0:.1f}s]")

---
## 실험 완료 체크리스트

- [ ] `notebooks/week5_hybrid_search.ipynb` (이 파일)
- [ ] `docs/week5_retrospective.md` — 회고 & ablation 결과표
- [ ] `docs/adr/week5_retrieval_strategy.md` — ADR
- [ ] `docs/imgs/week5_latency.png` — latency 차트

### 6주차 연결 포인트
Error Case 3개에서 확인한 실패 원인을 LangGraph의 **검색 → 품질 판단 → 재검색** 루프로 해결할 예정입니다.  
특히 context가 부족하다고 판단될 때 쿼리를 재작성(Multi-Query / HyDE)하는 노드를 추가하면  
이미지/표 기반 실패와 위치 기반 질문 실패를 부분적으로 커버할 수 있습니다.